# Ingesta de relaciones película-género

Lee `movie_genre.json` (JSON) desde Bronze, aplica un esquema explícito, estandariza las columnas y guarda el resultado como Parquet en `silver/movie_genre` (modo `overwrite`).

### Configuración

Carga las rutas de almacenamiento y las funciones comunes, y lee el parámetro `p_environment`.

In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
dbutils.widgets.text("p_environment", "dev", "environment")
v_env = dbutils.widgets.get("p_environment")

In [0]:
from pyspark.sql.functions import lit, col, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

### Lectura del archivo

Primero lee `movie_genre.json` con inferencia para inspeccionarlo y luego lo relee con un esquema explícito.

In [0]:
movie_genre_df = (
    spark.read
        .format("json")
        .option("header", "true")
        .load(f"{bronze_folder_path}/movie_genre.json")
)
movie_genre_df.show()

In [0]:
movie_genre_shema = StructType([
    StructField("genreId", IntegerType(), True),
    StructField("movieId", IntegerType(), True)
])
movie_genre_df = (
    spark.read
        .format("json")
        .schema(movie_genre_shema)
        .load(f"{bronze_folder_path}/movie_genre.json")
)
movie_genre_df.printSchema()
display(movie_genre_df)

### Transformación de columnas

Renombra `movieId` y `genreId` a snake_case y agrega `ingestion_date` y `env`.

In [0]:
movie_genre_df_final = (
    movie_genre_df
    .withColumnsRenamed({"genreId": "genre_id", "movieId": "movie_id"})
    .withColumns({"env": lit(v_env)})
)

movie_genre_df_final = add_ingestion_date(movie_genre_df_final)

display(movie_genre_df)

### Escritura en Silver

Guarda el resultado como Parquet en `silver/movie_genre` en modo `overwrite`.

In [0]:
(
    movie_genre_df_final
    .write
    .format("parquet")
    .mode("overwrite")
    #.partitionBy("movie_id")
    .save(f"{silver_folder_path}/movie_genre")
)

### Fin

Devuelve `Success` al notebook orquestador.

In [0]:
dbutils.notebook.exit("Success")